In [1]:
%%capture
!uv pip install numpy pandas scanpy

In [2]:
import numpy as np
import pandas as pd
import scanpy as sc

In [3]:
import warnings
import gc
warnings.simplefilter(action='ignore', category=Warning)

In [4]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
working_dir = "/content/drive/MyDrive/drosophila_scRNAseq_project"
os.chdir(working_dir)

In [6]:
os.listdir()

['data ', 'notebook', 'result ']

In [7]:
for root, dirs, files in os.walk("."):
    level = root.count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for file in files:
        print("  " * (level + 1) + file)

./
  data /
    raw/
      raw-20260829T152102Z-1-001/
        raw/
          ncov_pbmc_1.h5
          ncov_pbmc_15.h5
          normal_pbmc_14.h5
          normal_pbmc_13.h5
          normal_pbmc_19.h5
          ncov_pbmc_17.h5
          ncov_pbmc_16.h5
          normal_pbmc_5.h5
    integrated /
    filtered/
  notebook/
    nb01_ds_anndata.ipynb
  result /


In [9]:
os.listdir('data /raw')

['ncov_pbmc_15.h5',
 'normal_pbmc_5.h5',
 'ncov_pbmc_1.h5',
 'ncov_pbmc_17.h5',
 'normal_pbmc_13.h5',
 'normal_pbmc_14.h5',
 'ncov_pbmc_16.h5',
 'normal_pbmc_19.h5']

In [10]:
adata = sc.read_10x_h5("/content/drive/MyDrive/drosophila_scRNAseq_project/data /raw/ncov_pbmc_1.h5")

In [11]:
type(adata)

anndata._core.anndata.AnnData

In [12]:
adata

AnnData object with n_obs × n_vars = 1500 × 33538
    var: 'gene_ids', 'feature_types', 'genome'

In [13]:
adata.X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2005318 stored elements and shape (1500, 33538)>

In [14]:
adata.obs

""
AGGTAGGTCGTTGTTT-1
TAGAGTCGTCCTCCAT-1
CCCTGATAGCGAACTG-1
TCATCATTCCACGTAA-1
ATTTACCCAAGCCTGC-1
...
GGTTCTCAGGGATCGT-1
CCGATCTTCTATCGCC-1
GTAATCGGTAAGCTCT-1
TCATGAGGTGATCGTT-1


In [16]:
import os
import glob
import scanpy as sc

# Define directory where your raw .h5 files are stored
raw_dir = "/content/drive/MyDrive/drosophila_scRNAseq_project/data /raw"

# Find all .h5 files in that directory
h5_files = sorted(glob.glob(os.path.join(raw_dir, "*.h5")))

# Dictionary to hold individual loaded AnnData objects
adatas = {}

print(f"Found {len(h5_files)} .h5 files to load.")

for file_path in h5_files:
    # Get sample name from the filename (e.g., 'ncov_pbmc_1')
    sample_name = os.path.basename(file_path).replace(".h5", "")
    print(f"Loading {sample_name}...")

    # Load 10x H5 file
    adata = sc.read_10x_h5(file_path)

    # Make gene names unique to avoid downstream errors
    adata.var_names_make_unique()

    # Append sample name to cell barcodes to avoid overlap between samples
    adata.obs_names = [f"{sample_name}_{barcode}" for barcode in adata.obs_names]

    # Track origin sample in metadata
    adata.obs['sample'] = sample_name

    adatas[sample_name] = adata

# Combine all individual samples into one master AnnData object
adata = sc.concat(adatas, merge="same")
adata.obs_names_make_unique()

print("\n--- Integrated Master AnnData ---")
print(adata)

Found 8 .h5 files to load.
Loading ncov_pbmc_1...
Loading ncov_pbmc_15...
Loading ncov_pbmc_16...
Loading ncov_pbmc_17...
Loading normal_pbmc_13...
Loading normal_pbmc_14...
Loading normal_pbmc_19...
Loading normal_pbmc_5...

--- Integrated Master AnnData ---
AnnData object with n_obs × n_vars = 12000 × 33538
    obs: 'sample'
    var: 'gene_ids', 'feature_types', 'genome'


In [17]:
adata.obs

,sample
ncov_pbmc_1_AGGTAGGTCGTTGTTT-1,ncov_pbmc_1
ncov_pbmc_1_TAGAGTCGTCCTCCAT-1,ncov_pbmc_1
ncov_pbmc_1_CCCTGATAGCGAACTG-1,ncov_pbmc_1
ncov_pbmc_1_TCATCATTCCACGTAA-1,ncov_pbmc_1
ncov_pbmc_1_ATTTACCCAAGCCTGC-1,ncov_pbmc_1
...,...
normal_pbmc_5_TCGAACATCGTGGACC-5,normal_pbmc_5
normal_pbmc_5_CATTGCCGTGAATAAC-5,normal_pbmc_5
normal_pbmc_5_GTTATGGCATCGTGGC-5,normal_pbmc_5
normal_pbmc_5_CAGATCACATGTGCTA-5,normal_pbmc_5


In [18]:
adata.var

,gene_ids,feature_types,genome
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38
FAM138A,ENSG00000237613,Gene Expression,GRCh38
OR4F5,ENSG00000186092,Gene Expression,GRCh38
AL627309.1,ENSG00000238009,Gene Expression,GRCh38
AL627309.3,ENSG00000239945,Gene Expression,GRCh38
...,...,...,...
AC233755.2,ENSG00000277856,Gene Expression,GRCh38
AC233755.1,ENSG00000275063,Gene Expression,GRCh38
AC240274.1,ENSG00000271254,Gene Expression,GRCh38
AC213203.1,ENSG00000277475,Gene Expression,GRCh38


In [19]:
adata.var.head()

,gene_ids,feature_types,genome
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38
FAM138A,ENSG00000237613,Gene Expression,GRCh38
OR4F5,ENSG00000186092,Gene Expression,GRCh38
AL627309.1,ENSG00000238009,Gene Expression,GRCh38
AL627309.3,ENSG00000239945,Gene Expression,GRCh38


In [20]:
adata.layers['raw_counts'] = adata.X.copy()

In [21]:
adata.uns

OrderedDict()

In [22]:
adata.obsm

AxisArrays with keys: 